# Phase 2: Data Acquisition

Proses pengumpulan data sekuens protein dari UniProt. Kita menargetkan 6 famili/fungsi protein utama menggunakan data yang sudah direview (Swiss-Prot).

**Target Fungsi/Famili:**
1. Kinase
2. GPCR (G-protein coupled receptor)
3. Ion Channel
4. Transcription Factor (Transcription regulation)
5. Hydrolase
6. Oxidoreductase

In [ ]:
import requests
import pandas as pd
import os
from io import StringIO
import matplotlib.pyplot as plt
import seaborn as sns
import time

# Membuat direktori untuk menyimpan data mentah
os.makedirs('data/raw', exist_ok=True)

In [ ]:
# Konfigurasi target pencarian berdasarkan keyword UniProt
targets = {
    'Kinase': 'keyword:"Kinase"',
    'GPCR': 'keyword:"G-protein coupled receptor"',
    'Ion Channel': 'keyword:"Ion channel"',
    'Transcription Factor': 'keyword:"Transcription regulation"',
    'Hydrolase': 'keyword:"Hydrolase"',
    'Oxidoreductase': 'keyword:"Oxidoreductase"'
}

base_url = "https://rest.uniprot.org/uniprotkb/search"
all_data = []
max_per_class = 5000  # Mengatasi class imbalance dan membatasi ukuran dataset total

print("Memulai proses fetch data dari UniProt...")
for family, query in targets.items():
    print(f"Fetching {family}...")
    params = {
        "query": f"(reviewed:true) AND ({query})",
        "format": "tsv",
        "fields": "accession,id,protein_name,length,sequence",
        "size": 500
    }
    
    family_data = []
    url = base_url
    
    while url and sum(len(df) for df in family_data) < max_per_class:
        response = requests.get(url, params=params if url == base_url else None)
        if response.status_code == 200:
            df = pd.read_csv(StringIO(response.text), sep='\\t')
            df = df.dropna(subset=['Sequence'])
            family_data.append(df)
            
            link_header = response.headers.get('Link')
            url = None
            if link_header and 'rel="next"' in link_header:
                import re
                match = re.search(r'<([^>]+)>;\\s*rel="next"', link_header)
                if match:
                    url = match.group(1)
        else:
            print(f"  Gagal mengambil data. Status code: {response.status_code}")
            break
        
        time.sleep(0.5)

    if family_data:
        family_df = pd.concat(family_data, ignore_index=True)
        if len(family_df) > max_per_class:
            family_df = family_df.sample(n=max_per_class, random_state=42)
        family_df['Family'] = family
        all_data.append(family_df)
        print(f"  Berhasil memproses {len(family_df)} sekuens untuk {family}.")

# Menggabungkan semua dataframe dari setiap kelas
final_df = pd.concat(all_data, ignore_index=True)

# Mengganti nama kolom agar konsisten dan mudah diakses
final_df.columns = ['Entry', 'Entry_Name', 'Protein_Name', 'Length', 'Sequence', 'Family']

# Menyimpan hasil akuisisi data ke dalam file CSV
output_path = "data/raw/protein_sequences.csv"
final_df.to_csv(output_path, index=False)
print(f"\nData berhasil disimpan ke {output_path}. Total records: {len(final_df)}")

In [ ]:
# Visualisasi Distribusi Kelas untuk memastikan data cukup seimbang (balanced)
plt.figure(figsize=(10, 6))
sns.countplot(data=final_df, y='Family', order=final_df['Family'].value_counts().index, palette='viridis')
plt.title('Distribusi Famili Protein dalam Dataset (Maks. 5000 per kelas)')
plt.xlabel('Jumlah Sekuens')
plt.ylabel('Famili/Fungsi Protein')
plt.tight_layout()
plt.show()

# Tampilkan statistik ringkas
print("Statistik Jumlah Sekuens per Kelas:")
print(final_df['Family'].value_counts())
print(f"\nTotal Keseluruhan Dataset: {len(final_df)} baris")